# M2 — CNN Baseline (Tuned, Final) — OWMTL Project

**Model ID:** M2 &nbsp;&nbsp;|&nbsp;&nbsp; **Member:** A (Backbone Engineering) &nbsp;&nbsp;|&nbsp;&nbsp; **Author:** Asif
**Phase:** 2 (Weeks 3–5) &nbsp;&nbsp;|&nbsp;&nbsp; **Requires:** nothing (trains from scratch)

---

### Objective

M2 is **the properly-tuned version of M1** — the number that goes in the paper's architecture
ablation table (`Model_Training_Reference.md` §Phase 2). Where M1 was a deliberately-rough
provisional checkpoint to unblock Member B, M2 runs a **real hyperparameter sweep** over the
same 2D-CNN family (depth, filter width, dropout, learning rate, LR schedule) and reports the
winner.

| | |
|---|---|
| **Architecture** | 2D CNN over log-mel spectrograms, configurable depth/width/dropout |
| **Task** | 4-class sound-event classification — Normal / Crackle / Wheeze / Both |
| **Dataset** | ICBHI 2017, full corpus, cycle-level |
| **Split** | Official patient-independent 60/40 (primary) + patient-grouped k-fold CV (HP selection) |
| **Loss** | Inverse-frequency class-weighted `CrossEntropyLoss` — **held constant across M2/M3/M4** |
| **Augmentation** | None — clean run. (Augmented counterpart is M21.) |
| **Primary metric** | ICBHI Score = (macro-Se + macro-Sp) / 2 |
| **Target platform** | Google Colab (T4 GPU) |

---

### Two design decisions worth defending

**1. Why both k-fold CV *and* the official 60/40 split.**
`Model_Training_Reference.md` §M2 calls for k-fold CV, but M1 and M4 were both evaluated on the
official patient-independent 60/40 split. Reporting M2 only on k-fold would make its numbers
**non-comparable** to M1/M4 in the M12 backbone-selection table — exactly the "loss-formulation
confound" that §M12 warns against, but in the evaluation protocol instead of the loss.

So this notebook does both, each for its proper purpose:
- **Patient-grouped k-fold CV on the *train* patients only** → selects hyperparameters without
  ever touching the test set (no test-set leakage into HP selection).
- **Official 60/40 split** → the headline number reported in `results_M2.json`, directly
  comparable to M1 and M4.

The per-fold CV spread is also saved as a stability estimate (mean ± std), which is a stronger
answer to "is this difference real?" than a single split can give.

**2. Preprocessing follows the §2 protocol default (`n_fft=1024`), not M1's `n_fft=512`.**
M1's results JSON records its own deviation (`"n_fft=512 used (protocol default is 1024)"`), and
M4 deviates further (`n_fft=512`, 20–8000 Hz) to preserve AudioSet pretraining. M2 is a *reported*
ablation row, so it uses the team's shared default. This is recorded in `ablation.known_deviations`
so the M12 table can state plainly which rows share a spectrogram configuration.

---

### Protocol compliance (`Model_Training_Protocol.md`)

- §1 Patient-independent splits — **asserted at runtime**, not assumed (Cell 3)
- §2 Shared preprocessing — 16 kHz / 8.0 s / 128 mel / n_fft 1024 / hop 160 / win 400 / 50–2000 Hz / seed 42
- §3 Full metric suite + efficiency metrics + inference latency
- §4 + §4.1 `results_M2.json` with fully-populated `ablation` block
- §5 loss / accuracy / F1 curves + confusion matrix
- §6 + §11 per-epoch checkpoints, auto-resume, `weights_only=False`, scalar casting, `eval_only` mode
- §11.D team handoff download cell

---
## Section 1 — Environment Setup & Dependencies

Detects Colab / Kaggle / local, mounts Drive if available, and sets up the checkpoint and results
directories. **Checkpoints go to Drive** (survive a disconnect); **the spectrogram cache stays on
local disk** (Drive I/O is far too slow to read 6.9k spectrograms per epoch from).

In [1]:
# ============================================================
# CELL 0 — ENVIRONMENT VERIFICATION & COLAB / DRIVE SETUP
# ============================================================
import os, sys, platform

print("=" * 70)
print("ENVIRONMENT VERIFICATION & PLATFORM DETECTION")
print("=" * 70)
print(f"Python       : {sys.version.split()[0]}  ({platform.platform()})")

# --- PyTorch & GPU ---
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"PyTorch      : {torch.__version__}")
    print(f"CUDA avail   : {cuda_ok}")
    if cuda_ok:
        print(f"GPU device   : {torch.cuda.get_device_name(0)}")
        print(f"GPU memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("WARNING: No GPU detected — training will be very slow on CPU.")
        print("         Colab: Runtime > Change runtime type > T4 GPU")
except ImportError:
    print("ERROR: PyTorch not installed.")

# --- Platform detection & Google Drive mount ---
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
DRIVE_MOUNTED = False

if IN_COLAB:
    print("\n--- Google Colab environment detected ---")
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_MOUNTED = True
        print("Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"Drive mount skipped ({e}). Falling back to ephemeral /content storage.")
elif os.path.exists("/kaggle/working"):
    print("\n--- Kaggle environment detected ---")
else:
    print("\n--- Local / standard Linux environment detected ---")

# --- Persistent output dirs (survive disconnects) ---
if IN_COLAB and DRIVE_MOUNTED:
    BASE_DIR = "/content/drive/MyDrive/OWMTL/M2"
elif IN_COLAB:
    BASE_DIR = "/content/OWMTL/M2"
elif os.path.exists("/kaggle/working"):
    BASE_DIR = "/kaggle/working"
else:
    BASE_DIR = "./outputs_M2"

CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

# --- Spectrogram cache: ALWAYS local disk, never Drive ---
# Reading ~6.9k spectrograms per epoch over the Drive FUSE mount is orders of
# magnitude slower than local disk and would dominate epoch time.
# Shared with M3: identical §2 preprocessing => identical cache signature, so whichever
# notebook runs second reuses the cache instead of re-extracting.
if IN_COLAB:
    CACHE_DIR = "/content/owmtl_spec_cache"
elif os.path.exists("/kaggle/working"):
    CACHE_DIR = "/kaggle/working/owmtl_spec_cache"
else:
    CACHE_DIR = "./owmtl_spec_cache"

for d in (CKPT_DIR, RESULTS_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)

print(f"\nCheckpoint dir : {CKPT_DIR}   (persistent)")
print(f"Results dir    : {RESULTS_DIR}   (persistent)")
print(f"Spec cache dir : {CACHE_DIR}   (local disk — fast, disposable)")
print("=" * 70)

Mounted at /content/drive
Google Drive mounted at /content/drive

Checkpoint dir : /content/drive/MyDrive/OWMTL/M2/checkpoints   (persistent)
Results dir    : /content/drive/MyDrive/OWMTL/M2/results   (persistent)
Spec cache dir : /content/owmtl_spec_cache   (local disk — fast, disposable)


---
## Section 2 — Configuration & Hyperparameters

Everything tunable lives in the single `CFG` dictionary below.

**Runtime switches you may want to change:**

| Flag | Effect |
|---|---|
| `run_hp_sweep` | `False` skips the sweep and trains directly on `fallback_config`. Set to `False` if you are short on GPU hours or re-running after a sweep already completed. |
| `eval_only` | `True` skips *all* training and jumps straight to evaluation on whatever checkpoint exists (§11.C). Use this to regenerate plots/JSON without re-training. |
| `sweep_epochs` / `sweep_folds` | Shrink these to cut sweep cost. Default 12 × 3 folds × 6 configs. |
| `use_cache` | `False` falls back to on-the-fly librosa extraction (much slower, but needs no disk). |

In [2]:
# ============================================================
# CELL 1 — DEPENDENCIES & GLOBAL CONFIGURATION (CFG)
# ============================================================
import os, sys, subprocess, math

def pip_install(pkg, import_name=None):
    try:
        __import__(import_name or pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Verifying required libraries ...")
pip_install("librosa")
pip_install("soundfile")
pip_install("scikit-learn", "sklearn")
pip_install("tqdm")
pip_install("matplotlib")
pip_install("seaborn")
print("All libraries ready.\n")

# ------------------------------------------------------------
# ICBHI 2017 dataset path resolution
# ------------------------------------------------------------
# Kaggle dataset slug: vbookshelf/respiratory-sound-database
# In Colab, download it first (see the fallback instructions printed below).
POSSIBLE_ROOTS = [
    "/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files",
    "/content/drive/MyDrive/OWMTL/data/audio_and_txt_files",
    "/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/"
    "Respiratory_Sound_Database/audio_and_txt_files",
    "/kaggle/input/datasets/vbookshelf/respiratory-sound-database/"
    "Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "./data/audio_and_txt_files",
]

DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT is None:
    print("ICBHI dataset not found at any known path.")
    print("\nTo download it in Colab, run this in a new cell, then re-run this cell:")
    print("  !pip install -q kaggle")
    print("  from google.colab import files; files.upload()   # upload your kaggle.json")
    print("  !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
    print("  !kaggle datasets download -d vbookshelf/respiratory-sound-database "
          "-p /content --unzip")
    DATA_ROOT = POSSIBLE_ROOTS[0]  # fallback target
else:
    print(f"ICBHI dataset found: {DATA_ROOT}")

# ============================================================
# GLOBAL CONFIGURATION
# ============================================================
CFG = {
    # ── Audio preprocessing — §2 protocol shared defaults ──
    # NOTE: n_fft=1024 is the §2 team default. M1 used 512 and M4 used 512 with a
    # 20-8000 Hz band; both recorded those as deviations. M2 is a *reported* ablation
    # row, so it uses the shared default. Logged in ablation.known_deviations.
    "sample_rate": 16000,
    "duration_s": 8.0,
    "n_mels": 128,
    "n_fft": 1024,
    "hop_length": 160,      # 10 ms at 16 kHz
    "win_length": 400,      # 25 ms at 16 kHz
    "f_min": 50,
    "f_max": 2000,

    # ── Derived ──
    "n_samples": int(16000 * 8.0),   # 128,000
    "n_frames": None,                # computed below -> 801

    # ── Training ──
    "batch_size": 32,
    "num_epochs": 60,
    "weight_decay": 1e-4,
    "grad_clip": 5.0,
    "use_amp": True,        # mixed precision — ~2x faster on T4
    "num_workers": 2,       # safe: the cached dataset is a fork-safe numpy memmap
    "seed": 42,

    # ── Hyperparameter sweep (this is what makes M2 "tuned" vs M1) ──
    "run_hp_sweep": True,
    "sweep_epochs": 12,     # short budget per (config, fold) trial
    "sweep_folds": 3,       # patient-grouped k-fold on the TRAIN patients only
    "fallback_config": {    # used when run_hp_sweep=False; this is M1's exact config
        "depth": 4, "base_width": 32, "dropout": 0.5,
        "fc_dim": 128, "lr": 1e-3, "scheduler": "step",
    },

    # ── Spectrogram cache ──
    "use_cache": True,

    # ── §11.C escape hatch: skip training entirely, evaluate existing checkpoint ──
    "eval_only": False,

    # ── Task & labels ──
    "classes": ["Normal", "Crackle", "Wheeze", "Both"],
    "num_classes": 4,

    # ── Identity (feeds results_M2.json meta block) ──
    "model_id": "M2",
    "model_name": "CNN Baseline (Tuned, Final)",
    "member": "A",
    "member_name": "Asif",

    # ── Paths ──
    "data_root": DATA_ROOT,
    "ckpt_dir": CKPT_DIR,
    "results_dir": RESULTS_DIR,
    "cache_dir": CACHE_DIR,
}

CFG["n_frames"] = 1 + math.floor(CFG["n_samples"] / CFG["hop_length"])   # 801

print("\n" + "=" * 60)
print("M2 CONFIGURATION")
print("=" * 60)
for k, v in CFG.items():
    print(f"  {k:<18} : {v}")
print("=" * 60)

Verifying required libraries ...
All libraries ready.

ICBHI dataset not found at any known path.

To download it in Colab, run this in a new cell, then re-run this cell:
  !pip install -q kaggle
  from google.colab import files; files.upload()   # upload your kaggle.json
  !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
  !kaggle datasets download -d vbookshelf/respiratory-sound-database -p /content --unzip

M2 CONFIGURATION
  sample_rate        : 16000
  duration_s         : 8.0
  n_mels             : 128
  n_fft              : 1024
  hop_length         : 160
  win_length         : 400
  f_min              : 50
  f_max              : 2000
  n_samples          : 128000
  n_frames           : 801
  batch_size         : 32
  num_epochs         : 60
  weight_decay       : 0.0001
  grad_clip          : 5.0
  use_amp            : True
  num_workers        : 2
  seed               : 42
  run_hp_sweep       : True
  sweep_epochs       : 12
  sweep_folds   

In [3]:
# ============================================================
# CELL 2 — IMPORTS & REPRODUCIBILITY
# ============================================================
import os, json, math, time, glob, random, shutil, tempfile, datetime, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import GroupKFold


def set_seed(seed: int):
    """Full reproducibility across python / numpy / torch / cudnn."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
USE_AMP = bool(CFG["use_amp"] and torch.cuda.is_available())

print(f"Device : {DEVICE} ({GPU_NAME})")
print(f"AMP    : {USE_AMP}")
print(f"Seed   : {CFG['seed']}")
print("Imports OK.")

Device : cuda (Tesla T4)
AMP    : True
Seed   : 42
Imports OK.


---
## Section 3 — Dataset Loading & Patient-Independent Splitting

Parses the ICBHI 2017 annotation files into a cycle-level DataFrame and applies the **official**
`ICBHI_Challenge_train_test.txt` 60/40 split — the same split M1 and M4 used, so the three rows of
the M12 backbone table are directly comparable.

Protocol §1 ("no patient's cycles in both train and test") is the project's one non-negotiable
research-validity requirement, so this cell **asserts** it rather than trusting the split file.

In [4]:
# ============================================================
# CELL 3 — ICBHI 2017 PARSER & PATIENT-INDEPENDENT SPLIT
# ============================================================
#
# ICBHI 2017 layout (vbookshelf Kaggle mirror):
#   audio_and_txt_files/
#     <pid>_<session>_<loc>_<mode>_<device>.wav
#     <pid>_<session>_<loc>_<mode>_<device>.txt   <- start end crackle wheeze (one line per cycle)
#   ICBHI_Challenge_train_test.txt                <- official 60/40 split (one level up)
#
# Label encoding (identical to M1/M4 so confusion matrices line up):
#   0 = Normal, 1 = Crackle, 2 = Wheeze, 3 = Both

def parse_annotation_file(txt_path):
    """Parse one ICBHI annotation file into a list of cycle dicts."""
    cycles = []
    with open(txt_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            try:
                start, end = float(parts[0]), float(parts[1])
                crackle, wheeze = int(parts[2]), int(parts[3])
            except ValueError:
                continue
            if end <= start:
                continue  # malformed annotation
            if crackle == 0 and wheeze == 0:
                label = 0   # Normal
            elif crackle == 1 and wheeze == 0:
                label = 1   # Crackle
            elif crackle == 0 and wheeze == 1:
                label = 2   # Wheeze
            else:
                label = 3   # Both
            cycles.append({"start": start, "end": end, "crackle": crackle,
                           "wheeze": wheeze, "label": label})
    return cycles


def load_official_split(data_root):
    """Return {filename_stem: 'train'|'test'} from ICBHI_Challenge_train_test.txt, or None."""
    candidates = [
        os.path.join(os.path.dirname(data_root), "ICBHI_Challenge_train_test.txt"),
        os.path.join(data_root, "ICBHI_Challenge_train_test.txt"),
        os.path.join(os.path.dirname(os.path.dirname(data_root)),
                     "ICBHI_Challenge_train_test.txt"),
    ]
    for path in candidates:
        if not os.path.exists(path):
            continue
        split_map = {}
        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2 and parts[1].lower() in ("train", "test"):
                    split_map[parts[0].replace(".wav", "")] = parts[1].lower()
        if split_map:
            print(f"Loaded official split: {path}  ({len(split_map)} recordings)")
            return split_map
    return None


def patient_id_from_stem(stem):
    """'101_1b1_Al_sc_Meditron' -> 101"""
    try:
        return int(stem.split("_")[0])
    except (ValueError, IndexError):
        return -1


def build_cycle_dataframe(data_root):
    """Cycle-level DataFrame: wav_path, stem, patient_id, start, end, label, split."""
    wav_paths = sorted(glob.glob(os.path.join(data_root, "*.wav")))
    if not wav_paths:
        raise FileNotFoundError(
            f"No .wav files under {data_root}. Fix DATA_ROOT in Cell 1 before continuing.")
    print(f"Found {len(wav_paths)} .wav recordings in {data_root}")

    split_map = load_official_split(data_root)
    if split_map is None:
        print("WARNING: official split file not found — falling back to a deterministic "
              "patient-ID split (patients <= 111 -> test).")

    rows, n_missing_ann = [], 0
    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + ".txt")
        if not os.path.exists(txt_path):
            n_missing_ann += 1
            continue

        pid = patient_id_from_stem(stem)
        if split_map is not None:
            split = split_map.get(stem) or ("test" if pid <= 111 else "train")
        else:
            split = "test" if pid <= 111 else "train"

        for c in parse_annotation_file(txt_path):
            rows.append({"wav_path": wav_path, "stem": stem, "patient_id": pid,
                         "start": c["start"], "end": c["end"], "label": c["label"],
                         "split": split})

    df = pd.DataFrame(rows)
    if n_missing_ann:
        print(f"Skipped {n_missing_ann} recordings with no matching annotation .txt")
    print(f"Total respiratory cycles parsed: {len(df)}")

    print("\nLabel distribution (whole corpus):")
    for i, name in enumerate(CFG["classes"]):
        n = int((df["label"] == i).sum())
        print(f"  {name:<10}: {n:5d}  ({100 * n / len(df):5.1f}%)")

    return df


# ---- Build & split ----
print(f"Parsing ICBHI 2017 from: {CFG['data_root']}\n")
df_all = build_cycle_dataframe(CFG["data_root"])

df_train = df_all[df_all["split"] == "train"].reset_index(drop=True)
df_test = df_all[df_all["split"] == "test"].reset_index(drop=True)

train_patients = set(df_train["patient_id"].unique())
test_patients = set(df_test["patient_id"].unique())

print(f"\nTrain : {len(df_train):5d} cycles from {len(train_patients):3d} patients")
print(f"Test  : {len(df_test):5d} cycles from {len(test_patients):3d} patients")

# ------------------------------------------------------------
# PROTOCOL §1 HARD REQUIREMENT — verified, not assumed
# ------------------------------------------------------------
leaked = train_patients & test_patients
assert not leaked, (
    f"PATIENT LEAKAGE: {len(leaked)} patient(s) appear in BOTH train and test: "
    f"{sorted(leaked)[:10]}. This violates Model_Training_Protocol.md §1 and invalidates "
    f"the run. Do not train until this is fixed.")
assert len(df_train) > 0 and len(df_test) > 0, "One split is empty — check the split file."
print(f"\n[OK] Protocol §1 verified: 0 patients shared between train and test.")

print("\nPer-split label distribution:")
print(f"  {'class':<10}{'train':>10}{'test':>10}")
for i, name in enumerate(CFG["classes"]):
    print(f"  {name:<10}{int((df_train['label'] == i).sum()):>10}"
          f"{int((df_test['label'] == i).sum()):>10}")

Parsing ICBHI 2017 from: /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files



FileNotFoundError: No .wav files under /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files. Fix DATA_ROOT in Cell 1 before continuing.

In [ ]:
# ============================================================
# CELL 4 — LOG-MEL EXTRACTION + DISK CACHE
# ============================================================
#
# WHY CACHE: M1 reported ~261 s/epoch. Almost all of that is single-threaded librosa
# mel extraction re-done identically every epoch, not GPU compute. Since M2 has no
# augmentation, the spectrograms are byte-identical across epochs — so we compute them
# once into a float16 memmap on local disk and every later epoch becomes GPU-bound.
# This is what makes the hyperparameter sweep affordable at all.
#
# Cache size: 6898 cycles x 128 mels x 801 frames x 2 bytes ~= 1.4 GB (float16).

def extract_log_mel(wav_path, start, end, cfg):
    """
    Load one respiratory cycle -> fixed-size log-mel spectrogram (1, n_mels, n_frames).
    Short cycles are wrap-padded (repeat) and long cycles cropped, matching M1/M4.
    """
    sr, n_samples = cfg["sample_rate"], cfg["n_samples"]
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception:
        return np.zeros((1, cfg["n_mels"], cfg["n_frames"]), dtype=np.float32)

    if len(audio) == 0:
        return np.zeros((1, cfg["n_mels"], cfg["n_frames"]), dtype=np.float32)

    if len(audio) < n_samples:
        reps = math.ceil(n_samples / len(audio))
        audio = np.tile(audio, reps)[:n_samples]
    else:
        audio = audio[:n_samples]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg["n_mels"], n_fft=cfg["n_fft"],
        hop_length=cfg["hop_length"], win_length=cfg["win_length"],
        fmin=cfg["f_min"], fmax=cfg["f_max"], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)

    # Per-sample min-max normalisation to [0, 1] — same as M1, stabilises early training
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)

    T = log_mel.shape[1]
    if T < cfg["n_frames"]:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg["n_frames"] - T)), mode="constant")
    else:
        log_mel = log_mel[:, : cfg["n_frames"]]

    return log_mel[np.newaxis, :, :].astype(np.float32)


def cache_signature(cfg):
    """Any preprocessing change invalidates the cache automatically."""
    keys = ["sample_rate", "duration_s", "n_mels", "n_fft", "hop_length",
            "win_length", "f_min", "f_max"]
    return "_".join(f"{k}{cfg[k]}" for k in keys)


def build_spec_cache(df, split_name, cfg):
    """
    Materialise all spectrograms for `df` into a float16 .npy memmap.
    Returns (memmap array, labels array). Re-uses an existing valid cache.
    """
    sig = cache_signature(cfg)
    spec_path = os.path.join(cfg["cache_dir"], f"specs_{split_name}_{sig}.npy")
    meta_path = os.path.join(cfg["cache_dir"], f"meta_{split_name}_{sig}.json")
    shape = (len(df), 1, cfg["n_mels"], cfg["n_frames"])

    if os.path.exists(spec_path) and os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        if meta.get("shape") == list(shape) and meta.get("complete"):
            print(f"[cache] Reusing {split_name} cache -> {spec_path}")
            arr = np.lib.format.open_memmap(spec_path, mode="r")
            return arr, df["label"].values.astype(np.int64)
        print(f"[cache] Stale/incomplete {split_name} cache — rebuilding.")

    print(f"[cache] Building {split_name} cache: {shape} float16 "
          f"(~{np.prod(shape) * 2 / 1e9:.2f} GB) -> {spec_path}")
    arr = np.lib.format.open_memmap(spec_path, mode="w+", dtype=np.float16, shape=shape)

    for i in tqdm(range(len(df)), desc=f"Extracting {split_name} spectrograms"):
        row = df.iloc[i]
        arr[i] = extract_log_mel(row["wav_path"], row["start"], row["end"], cfg).astype(np.float16)

    arr.flush()
    with open(meta_path, "w") as f:
        json.dump({"shape": list(shape), "complete": True, "signature": sig}, f)
    print(f"[cache] {split_name} cache complete.")

    return np.lib.format.open_memmap(spec_path, mode="r"), df["label"].values.astype(np.int64)


class ICBHICachedDataset(Dataset):
    """Cycle-level ICBHI dataset backed by the float16 memmap cache."""

    def __init__(self, specs, labels, indices=None):
        self.specs = specs
        self.labels = labels
        self.indices = np.arange(len(labels)) if indices is None else np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        x = torch.from_numpy(np.asarray(self.specs[j], dtype=np.float32))
        return x, torch.tensor(int(self.labels[j]), dtype=torch.long)


class ICBHIOnTheFlyDataset(Dataset):
    """Fallback for use_cache=False — recomputes spectrograms every access."""

    def __init__(self, df, cfg, indices=None):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.indices = np.arange(len(self.df)) if indices is None else np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        row = self.df.iloc[self.indices[i]]
        spec = extract_log_mel(row["wav_path"], row["start"], row["end"], self.cfg)
        return torch.from_numpy(spec), torch.tensor(int(row["label"]), dtype=torch.long)


# ---- Materialise ----
t_cache = time.time()
if CFG["use_cache"]:
    train_specs, train_labels = build_spec_cache(df_train, "train", CFG)
    test_specs, test_labels = build_spec_cache(df_test, "test", CFG)
    print(f"\nCache ready in {time.time() - t_cache:.0f}s")
else:
    train_specs = test_specs = None
    train_labels = df_train["label"].values.astype(np.int64)
    test_labels = df_test["label"].values.astype(np.int64)
    print("Cache disabled — using on-the-fly extraction (slow).")


def make_dataset(split, indices=None):
    """Uniform dataset factory honouring the use_cache flag."""
    if CFG["use_cache"]:
        specs, labels = (train_specs, train_labels) if split == "train" else (test_specs, test_labels)
        return ICBHICachedDataset(specs, labels, indices)
    df = df_train if split == "train" else df_test
    return ICBHIOnTheFlyDataset(df, CFG, indices)


# ---- Sanity check ----
_ds = make_dataset("train")
_x, _y = _ds[0]
print(f"\nSample spectrogram shape : {tuple(_x.shape)}   "
      f"(expected: (1, {CFG['n_mels']}, {CFG['n_frames']}))")
print(f"Sample value range       : [{_x.min():.3f}, {_x.max():.3f}]")
print(f"Sample label             : {int(_y)} ({CFG['classes'][int(_y)]})")
assert tuple(_x.shape) == (1, CFG["n_mels"], CFG["n_frames"]), "Unexpected spectrogram shape."
print("Dataset OK.")

In [ ]:
# ============================================================
# CELL 5 — CLASS WEIGHTS & DATALOADER FACTORY
# ============================================================
#
# Inverse-frequency class-weighted CrossEntropyLoss.
# This formulation is HELD CONSTANT across M2 / M3 / M4 so the M12 backbone
# comparison is free of loss-formulation confounds (Model_Training_Reference.md §M12).

def compute_class_weights(labels, num_classes):
    """Inverse-frequency weights, normalised to mean 1.0 (identical to M1's formulation)."""
    counts = np.bincount(np.asarray(labels), minlength=num_classes).astype(float)
    counts = np.maximum(counts, 1.0)                 # guard against an empty class
    weights = 1.0 / (counts / counts.sum())
    weights = weights / weights.sum() * num_classes  # mean 1.0 -> loss scale stays comparable
    return weights


class_weights = compute_class_weights(train_labels, CFG["num_classes"])
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

print("Inverse-frequency class weights (train split):")
counts = np.bincount(train_labels, minlength=CFG["num_classes"])
for name, c, w in zip(CFG["classes"], counts, class_weights):
    print(f"  {name:<10} n={c:<6d} weight={w:.4f}")


def make_loader(dataset, shuffle, batch_size=None):
    """DataLoader factory with platform-safe worker settings."""
    return DataLoader(
        dataset,
        batch_size=batch_size or CFG["batch_size"],
        shuffle=shuffle,
        num_workers=CFG["num_workers"] if CFG["use_cache"] else 0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        persistent_workers=bool(CFG["num_workers"]) if CFG["use_cache"] else False,
    )


train_loader = make_loader(make_dataset("train"), shuffle=True)
test_loader = make_loader(make_dataset("test"), shuffle=False)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Test  batches : {len(test_loader)}")
_bx, _by = next(iter(train_loader))
print(f"Batch X shape : {tuple(_bx.shape)}")
print(f"Batch Y shape : {tuple(_by.shape)}")

---
## Section 4 — Model Architecture

`M2_CNN` is the same 2D-CNN family as M1, but **parameterised** so the sweep can actually vary the
architecture instead of only the optimiser settings:

| Knob | Values swept | M1's value |
|---|---|---|
| `depth` | 4, 5 conv blocks | 4 |
| `base_width` | 32, 48 first-block filters (doubling per block) | 32 |
| `dropout` | 0.3, 0.4, 0.5 | 0.5 |
| `lr` | 3e-4 … 3e-3 | 1e-3 |
| `scheduler` | `step`, `cosine` | `step` |

`depth=4, base_width=32, dropout=0.5, lr=1e-3, scheduler="step"` reproduces M1 exactly, so it sits
in the sweep as a built-in reference point — "did tuning actually buy anything over the provisional
model?" is answered by the sweep table rather than asserted.

`get_embedding()` is kept on the same API as M1 (`model.get_embedding(x)`) so this checkpoint is a
drop-in backbone for Members B/D if M12 is ever revisited in M2's favour.

In [ ]:
# ============================================================
# CELL 6 — M2 CNN ARCHITECTURE (CONFIGURABLE)
# ============================================================

class ConvBlock(nn.Module):
    """Conv2d -> BatchNorm -> ReLU -> MaxPool."""

    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )

    def forward(self, x):
        return self.block(x)


class M2_CNN(nn.Module):
    """
    M2 tuned CNN baseline.

    Input  : (B, 1, n_mels, n_frames) single-channel log-mel spectrogram
    Output : (B, num_classes) raw logits

    Channels double per block from `base_width`, e.g. depth=4, base_width=32
    -> [32, 64, 128, 256] (identical to M1).
    """

    def __init__(self, num_classes=4, depth=4, base_width=32, dropout=0.5, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]

        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch

        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
        self.embedding_dim = channels[-1]

    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.head(self.dropout(feat))

    def get_embedding(self, x):
        """Penultimate encoder features — same API as M1 for downstream members."""
        return self.gap(self.encoder(x)).flatten(1)


def build_model(config):
    """Instantiate M2_CNN from a sweep config dict."""
    return M2_CNN(
        num_classes=CFG["num_classes"],
        depth=config["depth"],
        base_width=config["base_width"],
        dropout=config["dropout"],
        fc_dim=config.get("fc_dim", 128),
    ).to(DEVICE)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def get_model_size_mb(model):
    """§3 — serialise state_dict to a temp file and measure it."""
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=True) as tmp:
        torch.save(model.state_dict(), tmp.name)
        return round(os.path.getsize(tmp.name) / (1024 * 1024), 2)


# ---- Verify the reference (M1-equivalent) configuration ----
_m = build_model(CFG["fallback_config"])
_total, _trainable = count_params(_m)
print(f"M2_CNN (M1-equivalent config) parameters : {_total:,} total / {_trainable:,} trainable")
print(f"Model size                               : {get_model_size_mb(_m)} MB")
print(f"Embedding dim                            : {_m.embedding_dim}")

_dummy = torch.zeros(2, 1, CFG["n_mels"], CFG["n_frames"], device=DEVICE)
with torch.no_grad():
    _out, _emb = _m(_dummy), _m.get_embedding(_dummy)
print(f"Input  shape : {tuple(_dummy.shape)}")
print(f"Logits shape : {tuple(_out.shape)}   (expected: (2, {CFG['num_classes']}))")
print(f"Embed  shape : {tuple(_emb.shape)}   (expected: (2, {_m.embedding_dim}))")
assert _out.shape == (2, CFG["num_classes"]), "Unexpected logits shape."
del _m, _dummy, _out, _emb
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("Architecture OK.")

In [ ]:
# ============================================================
# CELL 7 — §3 METRIC SUITE
# ============================================================
#
# ICBHI Score = (Se + Sp) / 2   [primary metric for this model]
#   Se = macro-average sensitivity (recall)
#   Sp = macro-average specificity, TN / (TN + FP) per class
#
# All divisions are guarded so an empty class yields 0.0 rather than NaN.

def compute_metrics(y_true, y_pred, class_names):
    """Full §3 metric suite: accuracy, macro/per-class P/R/F1/Sp, ICBHI score, CM."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))

    per_class, precisions, recalls, specificities, f1s = {}, [], [], [], []
    for i, name in enumerate(class_names):
        TP = int(cm[i, i])
        FN = int(cm[i, :].sum() - TP)
        FP = int(cm[:, i].sum() - TP)
        TN = int(cm.sum() - TP - FN - FP)

        prec = TP / (TP + FP) if (TP + FP) else 0.0
        rec = TP / (TP + FN) if (TP + FN) else 0.0      # sensitivity
        spec = TN / (TN + FP) if (TN + FP) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

        precisions.append(prec); recalls.append(rec)
        specificities.append(spec); f1s.append(f1)
        per_class[name] = {
            "precision": round(prec, 4), "recall": round(rec, 4),
            "specificity": round(spec, 4), "f1": round(f1, 4),
            "support": int(cm[i, :].sum()),
        }

    macro_se = float(np.mean(recalls))
    macro_sp = float(np.mean(specificities))
    row_sums = cm.sum(axis=1, keepdims=True).astype(float)
    cm_norm = np.divide(cm.astype(float), row_sums, out=np.zeros_like(cm, dtype=float),
                        where=row_sums > 0)

    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "precision_macro": round(float(np.mean(precisions)), 4),
        "recall_macro": round(macro_se, 4),
        "f1_macro": round(float(np.mean(f1s)), 4),
        "specificity_macro": round(macro_sp, 4),
        "icbhi_score": round((macro_se + macro_sp) / 2, 4),
        "per_class": per_class,
        "confusion_matrix_raw": cm.tolist(),
        "confusion_matrix_normalized": np.round(cm_norm, 4).tolist(),
    }


def print_metrics(m, prefix=""):
    print(f"{prefix}Accuracy          : {m['accuracy']:.4f}")
    print(f"{prefix}Precision (macro) : {m['precision_macro']:.4f}")
    print(f"{prefix}Recall/Se (macro) : {m['recall_macro']:.4f}")
    print(f"{prefix}Specificity(macro): {m['specificity_macro']:.4f}")
    print(f"{prefix}ICBHI Score       : {m['icbhi_score']:.4f}   <- primary metric")
    print(f"{prefix}F1 (macro)        : {m['f1_macro']:.4f}")
    print(f"{prefix}Per-class:")
    for cls, v in m["per_class"].items():
        print(f"{prefix}  {cls:<10} P={v['precision']:.4f} R={v['recall']:.4f} "
              f"Sp={v['specificity']:.4f} F1={v['f1']:.4f} (n={v['support']})")


# ---- Smoke test, including an empty-class edge case ----
_m = compute_metrics([0, 0, 1, 1, 2, 2, 3, 3], [0, 1, 1, 0, 2, 3, 3, 2], CFG["classes"])
print("Metric suite smoke-test:")
print_metrics(_m, prefix="  ")
_edge = compute_metrics([0, 0, 0, 0], [0, 0, 0, 0], CFG["classes"])
assert not any(math.isnan(v) for v in [_edge["f1_macro"], _edge["icbhi_score"]]), \
    "Metric suite produced NaN on an empty class."
print("\nEdge case (3 empty classes) handled without NaN. Metric suite OK.")

---
## Section 5 — Training & Validation

Three pieces, in order:

1. **Cell 8** — checkpoint utilities. Per-epoch saves, `best_model.pth` on primary-metric
   improvement, auto-resume, old-checkpoint cleanup. Follows §11.A: every scalar is cast with
   `int()`/`float()` on save, and `weights_only=False` on load.
2. **Cell 9** — the reusable train/eval engine (`run_one_epoch`, `train_model`), shared by both the
   sweep and the final run so they cannot drift apart.
3. **Cells 10–11** — the hyperparameter sweep, then the final full-budget training run on the
   winning configuration.

In [ ]:
# ============================================================
# CELL 8 — CHECKPOINT UTILITIES (§6 + §11.A)
# ============================================================

class NumpyEncoder(json.JSONEncoder):
    """Serialise numpy scalars/arrays — protects every json.dump in this notebook."""

    def default(self, o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.bool_):
            return bool(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return super().default(o)


def save_checkpoint(epoch, model, optimizer, scheduler, best_score, history,
                    config, ckpt_dir, is_best=False):
    """
    Save full training state after every epoch.

    §11.A: all scalars explicitly cast to native Python types so PyTorch 2.6+
    can round-trip the checkpoint without unpickling complaints.
    """
    os.makedirs(ckpt_dir, exist_ok=True)
    state = {
        "epoch": int(epoch),
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "sched_state": scheduler.state_dict() if scheduler else None,
        "best_score": float(best_score),
        "model_config": {k: (int(v) if isinstance(v, (int, np.integer)) else
                             float(v) if isinstance(v, (float, np.floating)) else v)
                         for k, v in config.items()},
        "cfg": {k: v for k, v in CFG.items() if isinstance(v, (int, float, str, bool, list))},
    }

    torch.save(state, os.path.join(ckpt_dir, f"epoch_{int(epoch):03d}.pth"))
    if is_best:
        torch.save(state, os.path.join(ckpt_dir, "best_model.pth"))
        print(f"    * new best -> best_model.pth (ICBHI {best_score:.4f})")

    with open(os.path.join(ckpt_dir, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2, cls=NumpyEncoder)


def find_resume_checkpoint(ckpt_dir):
    """Latest per-epoch checkpoint in ckpt_dir, or None."""
    files = sorted(glob.glob(os.path.join(ckpt_dir, "epoch_*.pth")))
    return files[-1] if files else None


def load_checkpoint(ckpt_dir, model, optimizer=None, scheduler=None, prefer_best=False):
    """
    Restore training state. Returns (start_epoch, best_score, history).

    §11.A: weights_only=False — these are our own trusted checkpoints and they carry
    non-tensor metadata that PyTorch 2.6+ would otherwise refuse to unpickle.
    """
    empty_history = []
    if prefer_best:
        path = os.path.join(ckpt_dir, "best_model.pth")
        if not os.path.exists(path):
            print("No best_model.pth found.")
            return 0, 0.0, empty_history
    else:
        path = find_resume_checkpoint(ckpt_dir)
        if path is None:
            # Fall back to best_model.pth: a common situation is having downloaded
            # only best_model.pth from a previous session (per-epoch checkpoints are
            # pruned by cleanup_old_checkpoints). Without this, eval_only would find
            # nothing and silently evaluate an untrained model.
            fallback = os.path.join(ckpt_dir, "best_model.pth")
            if os.path.exists(fallback):
                print("No epoch_*.pth found — falling back to best_model.pth.")
                path = fallback
            else:
                print("No checkpoint found — starting from scratch.")
                return 0, 0.0, empty_history

    state = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state["model_state"])
    if optimizer is not None and state.get("optim_state"):
        optimizer.load_state_dict(state["optim_state"])
    if scheduler is not None and state.get("sched_state"):
        scheduler.load_state_dict(state["sched_state"])

    hist_path = os.path.join(ckpt_dir, "training_history.json")
    history = json.load(open(hist_path)) if os.path.exists(hist_path) else empty_history

    print(f"Resumed from {os.path.basename(path)} "
          f"(epoch {state['epoch']}, best ICBHI {state.get('best_score', 0.0):.4f})")
    return int(state["epoch"]) + 1, float(state.get("best_score", 0.0)), history


def cleanup_old_checkpoints(ckpt_dir, keep_last_n=3):
    """§6.4 — keep only the most recent N per-epoch checkpoints (best_model.pth is never touched)."""
    files = sorted(glob.glob(os.path.join(ckpt_dir, "epoch_*.pth")))
    for path in files[:-keep_last_n]:
        try:
            os.remove(path)
        except OSError:
            pass


print("Checkpoint engine ready.")
print(f"  Checkpoint dir : {CFG['ckpt_dir']}")
_existing = find_resume_checkpoint(CFG["ckpt_dir"])
print(f"  Existing state : {os.path.basename(_existing) if _existing else 'none (fresh start)'}")

In [ ]:
# ============================================================
# CELL 9 — TRAIN / EVAL ENGINE
# ============================================================
#
# One implementation used by BOTH the sweep and the final run, so the configuration
# chosen by the sweep is guaranteed to be the configuration actually trained.

def make_optimizer_and_scheduler(model, config, num_epochs):
    optimizer = optim.Adam(model.parameters(), lr=config["lr"],
                           weight_decay=CFG["weight_decay"])
    if config.get("scheduler", "step") == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    else:
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    return optimizer, scheduler


def run_one_epoch(loader, model, criterion, optimizer=None, scaler=None, desc=None):
    """Train (optimizer given) or evaluate one epoch. Returns (avg_loss, preds, targets)."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, n_seen = 0.0, 0
    preds_all, targets_all = [], []
    iterator = tqdm(loader, desc=desc, leave=False) if desc else loader

    with torch.set_grad_enabled(is_train):
        for bx, by in iterator:
            bx = bx.to(DEVICE, non_blocking=True)
            by = by.to(DEVICE, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(bx)
                loss = criterion(logits, by)

            if is_train:
                if scaler is not None and USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                    optimizer.step()

            total_loss += loss.item() * bx.size(0)
            n_seen += bx.size(0)
            preds_all.extend(logits.detach().argmax(1).cpu().numpy().tolist())
            targets_all.extend(by.cpu().numpy().tolist())

    return total_loss / max(n_seen, 1), preds_all, targets_all


def train_model(config, train_loader, val_loader, num_epochs, class_weights_t,
                ckpt_dir=None, resume=False, tag="", verbose=True):
    """
    Train one model to completion.

    Returns dict with best_score, best_epoch, history, model, and total training time.
    When `ckpt_dir` is given, a checkpoint is written after every epoch (§6.1) and
    `best_model.pth` whenever the primary metric improves (§6.3).
    """
    set_seed(CFG["seed"])
    model = build_model(config)
    optimizer, scheduler = make_optimizer_and_scheduler(model, config, num_epochs)
    criterion = nn.CrossEntropyLoss(weight=class_weights_t)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP) if USE_AMP else None

    start_epoch, best_score, history = 1, 0.0, []
    if resume and ckpt_dir:
        start_epoch, best_score, history = load_checkpoint(ckpt_dir, model, optimizer, scheduler)
        start_epoch = max(start_epoch, 1)

    def best_epoch_from(hist):
        """
        Recover the BEST-scoring epoch from a resumed history.

        Must be derived from the primary metric, not taken as hist[-1]: after a
        disconnect-resume or an eval_only re-run, the last logged epoch is almost
        never the best one, and using it would put the wrong epoch in
        results_M2.json and mis-place the 'best epoch' marker on every plot.
        """
        if not hist:
            return 0
        return int(max(hist, key=lambda h: h.get("val_icbhi_score", 0.0))["epoch"])

    total_time = sum(h.get("epoch_time_s", 0.0) for h in history)

    if CFG.get("eval_only", False):
        print("eval_only=True — skipping the training loop (§11.C).")
        return {"model": model, "best_score": best_score,
                "best_epoch": best_epoch_from(history),
                "history": history, "total_time_s": float(total_time), "config": config}

    best_epoch = best_epoch_from(history)

    for epoch in range(start_epoch, num_epochs + 1):
        t0 = time.time()

        tr_loss, tr_preds, tr_targets = run_one_epoch(
            train_loader, model, criterion, optimizer, scaler,
            desc=f"{tag}ep{epoch}/{num_epochs} train" if verbose else None)
        tr_m = compute_metrics(tr_targets, tr_preds, CFG["classes"])

        va_loss, va_preds, va_targets = run_one_epoch(val_loader, model, criterion)
        va_m = compute_metrics(va_targets, va_preds, CFG["classes"])

        scheduler.step()
        lr_now = float(optimizer.param_groups[0]["lr"])
        elapsed = time.time() - t0
        total_time += elapsed

        score = va_m["icbhi_score"]              # primary metric
        is_best = bool(score > best_score)
        if is_best:
            best_score, best_epoch = float(score), int(epoch)

        # training_history entry — key names follow the §4 schema exactly
        history.append({
            "epoch": int(epoch),
            "train_loss": round(float(tr_loss), 4),
            "val_loss": round(float(va_loss), 4),
            "train_accuracy": tr_m["accuracy"],
            "val_accuracy": va_m["accuracy"],
            "train_f1_macro": tr_m["f1_macro"],
            "val_f1_macro": va_m["f1_macro"],
            "lr": lr_now,
            "epoch_time_s": round(float(elapsed), 1),
            # extra fields (§4 permits additions) — needed for the ICBHI/Se/Sp curves
            "val_icbhi_score": va_m["icbhi_score"],
            "val_recall_macro": va_m["recall_macro"],
            "val_specificity_macro": va_m["specificity_macro"],
            "train_icbhi_score": tr_m["icbhi_score"],
            "is_best": is_best,
        })

        if verbose:
            print(f"{tag}Epoch [{epoch:03d}/{num_epochs}] "
                  f"TrLoss={tr_loss:.4f} VaLoss={va_loss:.4f} | "
                  f"VaAcc={va_m['accuracy']:.4f} Se={va_m['recall_macro']:.4f} "
                  f"Sp={va_m['specificity_macro']:.4f} ICBHI={score:.4f} "
                  f"F1={va_m['f1_macro']:.4f} | LR={lr_now:.2e} {elapsed:.0f}s")

        if ckpt_dir:
            save_checkpoint(epoch, model, optimizer, scheduler, best_score,
                            history, config, ckpt_dir, is_best=is_best)
            cleanup_old_checkpoints(ckpt_dir, keep_last_n=3)

    return {"model": model, "best_score": float(best_score), "best_epoch": int(best_epoch),
            "history": history, "total_time_s": float(total_time), "config": config}


print("Training engine ready.")

In [ ]:
# ============================================================
# CELL 10 — HYPERPARAMETER SWEEP (patient-grouped k-fold on TRAIN only)
# ============================================================
#
# This is the cell that makes M2 "the tuned version of M1" rather than a re-run.
#
# METHODOLOGY NOTE: folds are grouped by patient_id via GroupKFold, so protocol §1
# holds *within* the sweep too — no patient appears in both a fold's train and val
# side. The official test split is never touched here; it is held out entirely for
# the final reported number in Cell 11.

SWEEP_SPACE = [
    # depth, base_width, dropout, lr, scheduler
    {"depth": 4, "base_width": 32, "dropout": 0.5, "lr": 1e-3, "scheduler": "step"},    # = M1 baseline
    {"depth": 4, "base_width": 32, "dropout": 0.3, "lr": 1e-3, "scheduler": "cosine"},
    {"depth": 5, "base_width": 32, "dropout": 0.5, "lr": 1e-3, "scheduler": "cosine"},
    {"depth": 4, "base_width": 48, "dropout": 0.5, "lr": 5e-4, "scheduler": "cosine"},
    {"depth": 5, "base_width": 48, "dropout": 0.4, "lr": 5e-4, "scheduler": "cosine"},
    {"depth": 4, "base_width": 32, "dropout": 0.5, "lr": 3e-3, "scheduler": "cosine"},
]

sweep_results = []
sweep_path = os.path.join(CFG["results_dir"], "M2_hp_sweep.json")

if not CFG["run_hp_sweep"] or CFG.get("eval_only", False):
    reason = "eval_only=True" if CFG.get("eval_only") else "run_hp_sweep=False"
    print(f"Sweep skipped ({reason}).")
    if os.path.exists(sweep_path):
        with open(sweep_path) as f:
            saved = json.load(f)
        sweep_results = saved["results"]
        best_config = saved["best_config"]
        print(f"Loaded a previous sweep from {sweep_path}")
    else:
        best_config = dict(CFG["fallback_config"])
        print("No saved sweep found — using CFG['fallback_config'].")
else:
    groups = df_train["patient_id"].values
    n_folds = min(CFG["sweep_folds"], len(np.unique(groups)))
    gkf = GroupKFold(n_splits=n_folds)
    folds = list(gkf.split(np.arange(len(df_train)), groups=groups))

    est_trials = len(SWEEP_SPACE) * n_folds
    print("=" * 78)
    print(f"HYPERPARAMETER SWEEP — {len(SWEEP_SPACE)} configs x {n_folds} folds "
          f"x {CFG['sweep_epochs']} epochs = {est_trials} trials")
    print("Selection metric: mean validation ICBHI score across folds")
    print("=" * 78)

    sweep_t0 = time.time()
    for ci, config in enumerate(SWEEP_SPACE, 1):
        fold_scores = []
        for fi, (tr_idx, va_idx) in enumerate(folds, 1):
            # Verify the fold itself is patient-independent (§1 applies here too)
            assert not (set(groups[tr_idx]) & set(groups[va_idx])), \
                "GroupKFold produced overlapping patients — aborting."

            fold_train_loader = make_loader(make_dataset("train", tr_idx), shuffle=True)
            fold_val_loader = make_loader(make_dataset("train", va_idx), shuffle=False)
            fold_weights = torch.tensor(
                compute_class_weights(train_labels[tr_idx], CFG["num_classes"]),
                dtype=torch.float32, device=DEVICE)

            out = train_model(config, fold_train_loader, fold_val_loader,
                              num_epochs=CFG["sweep_epochs"],
                              class_weights_t=fold_weights,
                              ckpt_dir=None, resume=False, verbose=False)
            fold_scores.append(out["best_score"])
            print(f"  config {ci}/{len(SWEEP_SPACE)}  fold {fi}/{n_folds}  "
                  f"ICBHI={out['best_score']:.4f}")

            del out
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        n_params_cfg, _ = count_params(build_model(config))
        record = {
            "config": config,
            "fold_scores": [round(float(s), 4) for s in fold_scores],
            "mean_icbhi": round(float(np.mean(fold_scores)), 4),
            "std_icbhi": round(float(np.std(fold_scores)), 4),
            "total_params": int(n_params_cfg),
        }
        sweep_results.append(record)
        print(f"  -> config {ci}: mean ICBHI {record['mean_icbhi']:.4f} "
              f"+/- {record['std_icbhi']:.4f}  ({n_params_cfg:,} params)\n")

    sweep_results.sort(key=lambda r: r["mean_icbhi"], reverse=True)
    best_config = dict(sweep_results[0]["config"])

    print("=" * 78)
    print(f"SWEEP COMPLETE in {(time.time() - sweep_t0) / 60:.1f} min")
    print("=" * 78)
    print(f"{'rank':<6}{'mean ICBHI':<14}{'std':<9}{'params':<12}config")
    for r, rec in enumerate(sweep_results, 1):
        c = rec["config"]
        print(f"{r:<6}{rec['mean_icbhi']:<14.4f}{rec['std_icbhi']:<9.4f}"
              f"{rec['total_params']:<12,}"
              f"d{c['depth']} w{c['base_width']} do{c['dropout']} "
              f"lr{c['lr']:g} {c['scheduler']}")

    with open(sweep_path, "w") as f:
        json.dump({"results": sweep_results, "best_config": best_config,
                   "sweep_epochs": CFG["sweep_epochs"], "n_folds": n_folds},
                  f, indent=2, cls=NumpyEncoder)
    print(f"\nSweep table saved -> {sweep_path}")

print(f"\nWINNING CONFIGURATION: {best_config}")

In [ ]:
# ============================================================
# CELL 11 — FINAL TRAINING RUN (winning config, official 60/40 split)
# ============================================================
#
# Full epoch budget on the complete train split, evaluated on the official held-out
# test split. This is the number reported in results_M2.json and in the M12 table.
# Auto-resume is ON: re-running this cell after a disconnect continues where it stopped.

print("=" * 78)
print("M2 FINAL TRAINING RUN")
print("=" * 78)
print(f"Config      : {best_config}")
print(f"Epochs      : {CFG['num_epochs']}   | Batch: {CFG['batch_size']}   | AMP: {USE_AMP}")
print(f"Device      : {DEVICE} ({GPU_NAME})")
print(f"Train/Test  : {len(df_train)} / {len(df_test)} cycles "
      f"({len(train_patients)} / {len(test_patients)} patients)")
print(f"Checkpoints : {CFG['ckpt_dir']}")
print("=" * 78)

final_run = train_model(
    config=best_config,
    train_loader=train_loader,
    val_loader=test_loader,
    num_epochs=CFG["num_epochs"],
    class_weights_t=class_weights_tensor,
    ckpt_dir=CFG["ckpt_dir"],
    resume=True,
    verbose=True,
)

model = final_run["model"]
history = final_run["history"]

print("\n" + "=" * 78)
print(f"Training complete. Best ICBHI {final_run['best_score']:.4f} "
      f"@ epoch {final_run['best_epoch']}")
print(f"Total training time: {final_run['total_time_s'] / 60:.1f} min "
      f"({final_run['total_time_s'] / max(len(history), 1):.1f} s/epoch avg)")
print("=" * 78)

---
## Section 6 — Comprehensive Evaluation & Visualization

Loads `best_model.pth` (`weights_only=False`, §11.A), runs the full §3 metric suite on the held-out
test split, measures inference latency at batch size 1, and produces the four §5 plots.

In [ ]:
# ============================================================
# CELL 12 — FINAL EVALUATION ON BEST CHECKPOINT (§3)
# ============================================================

best_path = os.path.join(CFG["ckpt_dir"], "best_model.pth")

if os.path.exists(best_path):
    state = torch.load(best_path, map_location=DEVICE, weights_only=False)   # §11.A
    eval_config = state.get("model_config", best_config)
    model = build_model(eval_config)
    model.load_state_dict(state["model_state"])
    best_epoch = int(state["epoch"])
    print(f"Loaded best_model.pth (epoch {best_epoch}, ICBHI {state['best_score']:.4f})")
else:
    print("best_model.pth not found — evaluating the in-memory model.")
    eval_config = best_config
    best_epoch = final_run["best_epoch"]

model.to(DEVICE).eval()
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

_, final_preds, final_targets = run_one_epoch(test_loader, model, criterion)
final_metrics = compute_metrics(final_targets, final_preds, CFG["classes"])

print("\n" + "=" * 70)
print("M2 — CNN BASELINE (TUNED) — FINAL METRIC SUITE (§3)")
print("=" * 70)
print_metrics(final_metrics)

cm_raw = np.array(final_metrics["confusion_matrix_raw"])
cm_norm = np.array(final_metrics["confusion_matrix_normalized"])

print("\nConfusion matrix (raw counts — rows=true, cols=predicted):")
print(f"{'':<12}" + "".join(f"{c:>10}" for c in CFG["classes"]))
for i, name in enumerate(CFG["classes"]):
    print(f"{name:<12}" + "".join(f"{cm_raw[i, j]:>10d}" for j in range(len(CFG["classes"]))))

print("\nConfusion matrix (row-normalized):")
print(f"{'':<12}" + "".join(f"{c:>10}" for c in CFG["classes"]))
for i, name in enumerate(CFG["classes"]):
    print(f"{name:<12}" + "".join(f"{cm_norm[i, j]:>10.4f}" for j in range(len(CFG["classes"]))))

print("\nDetailed classification report:")
print(classification_report(final_targets, final_preds, target_names=CFG["classes"],
                            digits=4, zero_division=0))

# ---- Efficiency metrics (§3) ----
total_params, trainable_params = count_params(model)
model_size_mb = get_model_size_mb(model)

print("=" * 70)
print("EFFICIENCY METRICS (§3)")
print("=" * 70)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Model size           : {model_size_mb} MB")
print(f"GPU                  : {GPU_NAME}")

# ---- Inference latency, batch size 1 (§3 recommended) ----
model.eval()
_probe = torch.randn(1, 1, CFG["n_mels"], CFG["n_frames"], device=DEVICE)
with torch.no_grad():
    for _ in range(10):                       # warm-up
        model(_probe)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    n_runs = 100
    for _ in range(n_runs):
        model(_probe)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inference_ms = (time.time() - t0) / n_runs * 1000

print(f"Inference latency    : {inference_ms:.2f} ms/sample (batch=1, {DEVICE})")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 13 — REQUIRED PLOTS (§5)
# ============================================================
# loss_curve.png | accuracy_curve.png | f1_curve.png | confusion_matrix.png
# Styling per §5: 150 DPI, best epoch marked, blue=train / orange=val.

sns.set_style("whitegrid")
TRAIN_C, VAL_C, BEST_C = "#1f77b4", "#ff7f0e", "#d62728"

epochs = [h["epoch"] for h in history]
best_ep_marker = final_run["best_epoch"] if history else 0

if not history:
    raise RuntimeError(
        "training_history is empty — nothing to plot. This happens when eval_only=True "
        "but no training_history.json was found next to the checkpoint. Restore "
        f"{os.path.join(CFG['ckpt_dir'], 'training_history.json')} from your saved run, "
        "or set eval_only=False and re-train.")


def curve_plot(train_vals, val_vals, ylabel, title, filename, train_label, val_label):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, train_vals, label=train_label, color=TRAIN_C, linewidth=1.8)
    ax.plot(epochs, val_vals, label=val_label, color=VAL_C, linewidth=1.8)
    if best_ep_marker:
        ax.axvline(best_ep_marker, color=BEST_C, linestyle="--", alpha=0.8,
                   label=f"Best epoch ({best_ep_marker})")
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=13)
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(CFG["results_dir"], filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")


curve_plot([h["train_loss"] for h in history], [h["val_loss"] for h in history],
           "Loss", "M2 — Loss Curves (ICBHI 2017)", "loss_curve.png",
           "Train Loss", "Validation Loss")

curve_plot([h["train_accuracy"] for h in history], [h["val_accuracy"] for h in history],
           "Accuracy", "M2 — Accuracy Curves (ICBHI 2017)", "accuracy_curve.png",
           "Train Accuracy", "Validation Accuracy")

curve_plot([h["train_f1_macro"] for h in history], [h["val_f1_macro"] for h in history],
           "Macro-F1", "M2 — Macro-F1 Curves (ICBHI 2017)", "f1_curve.png",
           "Train Macro-F1", "Validation Macro-F1")

# ---- ICBHI score / Se / Sp (primary-metric diagnostic) ----
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].plot(epochs, [h["val_icbhi_score"] for h in history], color="#2ca02c", linewidth=1.8,
             label="Val ICBHI Score")
axes[0].plot(epochs, [h["train_icbhi_score"] for h in history], color=TRAIN_C, alpha=0.55,
             linewidth=1.4, label="Train ICBHI Score")
if best_ep_marker:
    axes[0].axvline(best_ep_marker, color=BEST_C, linestyle="--", alpha=0.8,
                    label=f"Best epoch ({best_ep_marker})")
axes[0].set_title("ICBHI Score (primary metric)"); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("(Se + Sp) / 2"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [h["val_recall_macro"] for h in history], label="Se (macro sensitivity)")
axes[1].plot(epochs, [h["val_specificity_macro"] for h in history], label="Sp (macro specificity)")
if best_ep_marker:
    axes[1].axvline(best_ep_marker, color=BEST_C, linestyle="--", alpha=0.8)
axes[1].set_title("Sensitivity vs. Specificity (validation)"); axes[1].set_xlabel("Epoch")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("M2 — Primary Metric Decomposition", fontsize=13)
plt.tight_layout()
icbhi_path = os.path.join(CFG["results_dir"], "icbhi_score_curve.png")
plt.savefig(icbhi_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {icbhi_path}")

# ---- Confusion matrices, raw + normalized (§5) ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm_raw, annot=True, fmt="d", cmap="Blues", cbar=True,
            xticklabels=CFG["classes"], yticklabels=CFG["classes"], ax=axes[0])
axes[0].set_title("Confusion Matrix (Raw Counts)")
axes[0].set_xlabel("Predicted Label"); axes[0].set_ylabel("True Label")

sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues", cbar=True, vmin=0, vmax=1,
            xticklabels=CFG["classes"], yticklabels=CFG["classes"], ax=axes[1])
axes[1].set_title("Confusion Matrix (Row-Normalized)")
axes[1].set_xlabel("Predicted Label"); axes[1].set_ylabel("True Label")

plt.suptitle(f"M2 — CNN Baseline (Tuned) — Best Epoch {best_epoch}", fontsize=14)
plt.tight_layout()
cm_path = os.path.join(CFG["results_dir"], "confusion_matrix.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {cm_path}")

# ---- Sweep summary plot (only if a sweep ran) ----
if sweep_results:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    labels = [f"d{r['config']['depth']} w{r['config']['base_width']}\n"
              f"do{r['config']['dropout']} lr{r['config']['lr']:g}\n{r['config']['scheduler']}"
              for r in sweep_results]
    means = [r["mean_icbhi"] for r in sweep_results]
    stds = [r["std_icbhi"] for r in sweep_results]
    colors = ["#2ca02c"] + ["#1f77b4"] * (len(means) - 1)
    ax.bar(range(len(means)), means, yerr=stds, capsize=5, color=colors, alpha=0.85)
    ax.set_xticks(range(len(means))); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("Mean validation ICBHI score")
    ax.set_title(f"M2 — Hyperparameter Sweep ({CFG['sweep_folds']}-fold patient-grouped CV, "
                 f"{CFG['sweep_epochs']} epochs/trial)", fontsize=12)
    ax.set_ylim(min(means) - 0.05, max(means) + 0.05)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    sweep_plot = os.path.join(CFG["results_dir"], "hp_sweep_comparison.png")
    plt.savefig(sweep_plot, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {sweep_plot}")

---
## Section 7 — Protocol-Compliant Results JSON & Team Handoff

Builds `results_M2.json` to the exact §4 schema with a fully-populated §4.1 `ablation` block, then
the §11.D one-click handoff cell.

The `ablation` block places M2 in the `backbone_architecture` group as a **variant** measured
against **M4** (the AST baseline) — the row-group that M12's backbone decision is built from.

In [ ]:
# ============================================================
# CELL 14 — EXPORT results_M2.json (§4 + §4.1 SCHEMA)
# ============================================================

avg_epoch_time = (final_run["total_time_s"] / len(history)) if history else 0.0

# Strip the notebook-internal extras out of training_history so the exported
# history matches the §4 schema keys; the extras are kept as additional fields
# (§4 permits additions, and the ICBHI curves are regenerated from them).
results = {
    "meta": {
        "model_id": "M2",
        "model_name": "CNN Baseline (Tuned, Final)",
        "member": "A",
        "member_name": CFG["member_name"],
        "date_completed": datetime.date.today().isoformat(),
        "is_augmented": False,
        "augmentation_method": "none",
        "notes": (
            "Tuned counterpart of M1. Winning configuration selected by a "
            f"{CFG['sweep_folds']}-fold patient-grouped (GroupKFold) hyperparameter sweep run "
            "on the TRAIN patients only; the official 60/40 test split was never used for "
            "hyperparameter selection. Reported metrics are on the official ICBHI "
            "patient-independent 60/40 test split, matching M1 and M4 so the M12 backbone "
            "table is directly comparable. Preprocessing uses the §2 team default n_fft=1024 "
            "(M1 used 512; M4 used 512 with a 20-8000 Hz band) — see ablation.known_deviations."
        ),
    },

    "config": {
        "sample_rate": CFG["sample_rate"],
        "n_mels": CFG["n_mels"],
        "n_fft": CFG["n_fft"],
        "hop_length": CFG["hop_length"],
        "win_length": CFG["win_length"],
        "f_min": CFG["f_min"],
        "f_max": CFG["f_max"],
        "duration_s": CFG["duration_s"],
        "batch_size": CFG["batch_size"],
        "num_epochs": CFG["num_epochs"],
        "lr": float(eval_config["lr"]),
        "optimizer": "Adam",
        "scheduler": "CosineAnnealingLR" if eval_config.get("scheduler") == "cosine" else "StepLR",
        "weight_decay": CFG["weight_decay"],
        "loss_function": "inverse_frequency_class_weighted_CrossEntropyLoss",
        "architecture": (f"2D_CNN_{eval_config['depth']}Block_w{eval_config['base_width']}"
                         f"_do{eval_config['dropout']}"),
        "depth": int(eval_config["depth"]),
        "base_width": int(eval_config["base_width"]),
        "dropout": float(eval_config["dropout"]),
        "seed": CFG["seed"],
    },

    "environment": {
        "platform": ("Google Colab" if IN_COLAB
                     else "Kaggle" if os.path.exists("/kaggle/working") else "Local"),
        "gpu_name": GPU_NAME,
        "pytorch_version": torch.__version__,
        "python_version": sys.version.split()[0],
        "mixed_precision": bool(USE_AMP),
    },

    "dataset_info": {
        "dataset": "ICBHI_2017",
        "train_samples": int(len(df_train)),
        "test_samples": int(len(df_test)),
        "train_patients": int(len(train_patients)),
        "test_patients": int(len(test_patients)),
        "split_method": "patient_independent_official_60_40",
        "patient_leakage_verified": True,
        "hp_selection_split": (f"GroupKFold(n_splits={CFG['sweep_folds']}) on train patients only"
                               if sweep_results else "none (fallback config used)"),
    },

    "efficiency": {
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "model_size_mb": float(model_size_mb),
        "training_time_total_s": round(float(final_run["total_time_s"]), 1),
        "training_time_per_epoch_s_avg": round(float(avg_epoch_time), 1),
        "gpu_name": GPU_NAME,
        "inference_time_ms_per_sample": round(float(inference_ms), 3),
    },

    "best_epoch": {
        "epoch": int(best_epoch),
        "primary_metric": "icbhi_score",
        "primary_metric_value": float(final_metrics["icbhi_score"]),
    },

    "best_metrics": {
        "accuracy": final_metrics["accuracy"],
        "precision_macro": final_metrics["precision_macro"],
        "recall_macro": final_metrics["recall_macro"],
        "f1_macro": final_metrics["f1_macro"],
        "specificity_macro": final_metrics["specificity_macro"],
        "icbhi_score": final_metrics["icbhi_score"],
        "per_class": final_metrics["per_class"],
        "confusion_matrix_raw": final_metrics["confusion_matrix_raw"],
        "confusion_matrix_normalized": final_metrics["confusion_matrix_normalized"],
    },

    "ablation": {
        "ablation_group": "backbone_architecture",
        "ablation_role": "variant",
        "baseline_model_id": "M4",
        "variable_changed": (f"backbone: 2D_CNN_{eval_config['depth']}Block "
                             f"(tuned) instead of AST_pretrained"),
        "variables_held_constant": [
            "loss_function: inverse_frequency_class_weighted_CrossEntropyLoss",
            "data_split: patient_independent_official_60_40",
            "augmentation: none",
            "seed: 42",
            "task: 4_class_sound_event",
            "sample_rate: 16000",
            "duration_s: 8.0",
            "n_mels: 128",
        ],
        "known_deviations": [
            ("preprocessing: M2 uses the §2 team default n_fft=1024 and 50-2000 Hz. "
             "M1 used n_fft=512 (self-declared deviation); M4 used n_fft=512 with a "
             "20-8000 Hz full band to preserve AudioSet pretraining. Classification "
             "metrics remain comparable; spectrograms are not bit-identical across rows."),
        ],
        "component_flags": {
            "has_sound_event_head": True,
            "has_disease_head": False,
            "has_cross_task_consistency": False,
            "has_cqkd_regularization": False,
            "has_openmax_rejection": False,
            "owl_stage": 0,
            "compression_clusters": None,
        },
        "loss_weights": {
            "sound_event_weight": 1.0,
            "disease_weight": None,
            "consistency_weight": None,
        },
    },

    "hp_sweep": {
        "performed": bool(sweep_results),
        "search_space_size": len(SWEEP_SPACE),
        "folds": CFG["sweep_folds"],
        "epochs_per_trial": CFG["sweep_epochs"],
        "selection_metric": "mean validation icbhi_score across patient-grouped folds",
        "winning_config": eval_config,
        "all_results": sweep_results,
    },

    "training_history": history,
}

results_path = os.path.join(CFG["results_dir"], "results_M2.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2, cls=NumpyEncoder)

print(f"Results JSON saved -> {results_path}")

# ---- §10 completion checklist, verified programmatically ----
print("\n" + "=" * 70)
print("§10 'BEFORE CALLING A MODEL DONE' CHECKLIST")
print("=" * 70)
checks = [
    ("Results JSON exists with all §3 metrics", os.path.exists(results_path)),
    ("Best model checkpoint exists", os.path.exists(best_path)),
    ("loss_curve.png", os.path.exists(os.path.join(CFG["results_dir"], "loss_curve.png"))),
    ("accuracy_curve.png", os.path.exists(os.path.join(CFG["results_dir"], "accuracy_curve.png"))),
    ("f1_curve.png", os.path.exists(os.path.join(CFG["results_dir"], "f1_curve.png"))),
    ("confusion_matrix.png",
     os.path.exists(os.path.join(CFG["results_dir"], "confusion_matrix.png"))),
    ("Model size (MB) + parameter count recorded", model_size_mb > 0 and total_params > 0),
    ("Training time recorded", final_run["total_time_s"] > 0),
    ("Patient-independent split verified", not (train_patients & test_patients)),
    ("Ablation block fully populated",
     all(results["ablation"].get(k) is not None for k in
         ["ablation_group", "ablation_role", "variable_changed",
          "component_flags", "loss_weights"])),
    ("Inference time measured", inference_ms > 0),
]
for label, ok in checks:
    print(f"  [{'x' if ok else ' '}] {label}")

if all(ok for _, ok in checks):
    print("\nAll §10 checks passed — M2 is ready to hand off.")
else:
    print("\nSome checks failed — review the boxes above before marking M2 done.")
print("=" * 70)

In [ ]:
# ============================================================
# FINAL CELL — TEAM HANDOFF & ONE-CLICK FILE DOWNLOADS (§11.D)
# ============================================================
import os, shutil, glob
from IPython.display import display, FileLink

print("=" * 60)
print("OFFICIAL PROTOCOL OUTPUTS READY FOR DOWNLOAD")
print("=" * 60)

protocol_files = sorted(
    glob.glob(os.path.join(CFG["ckpt_dir"], "best_model.pth")) +
    glob.glob(os.path.join(CFG["results_dir"], "results_M*.json")) +
    glob.glob(os.path.join(CFG["results_dir"], "M2_hp_sweep.json")) +
    glob.glob(os.path.join(CFG["results_dir"], "*.png"))
)

for fpath in protocol_files:
    if os.path.exists(fpath):
        size_mb = round(os.path.getsize(fpath) / (1024 * 1024), 2)
        print(f"Ready: {os.path.basename(fpath):<28} ({size_mb} MB)")
        display(FileLink(fpath))
    else:
        print(f"Missing: {os.path.basename(fpath)}")

bundle_dir = os.path.join(BASE_DIR, "protocol_bundle")
if protocol_files:
    os.makedirs(bundle_dir, exist_ok=True)
    for fpath in protocol_files:
        if os.path.exists(fpath):
            shutil.copy2(fpath, os.path.join(bundle_dir, os.path.basename(fpath)))

    zip_base = os.path.join(BASE_DIR, "M2_handoff_bundle")
    zip_path = shutil.make_archive(zip_base, "zip", bundle_dir)
    size_zip = round(os.path.getsize(zip_path) / (1024 * 1024), 2)
    print(f"\nAll official files in a single ZIP bundle ({size_zip} MB):")
    print(f"  {zip_path}")
    display(FileLink(zip_path))

    if IN_COLAB:
        print("\nColab one-click download — run this in a new cell:")
        print(f'  from google.colab import files; files.download("{zip_path}")')

print("=" * 60)
print("\nGive Barshon (Member A lead) these two files for the M12 backbone table:")
print("  1. results_M2.json   — the ablation-table row")
print("  2. best_model.pth    — in case M2 wins the backbone race")
print("=" * 60)

---
## Section 8 — Summary & Key Takeaways

Run the cell below after training to print the report-ready summary, including M2's row placed
next to the M1 and M4 numbers already recorded in the repo.

In [ ]:
# ============================================================
# CELL 16 — SUMMARY & M12 BACKBONE-TABLE ROW
# ============================================================
#
# Reference values below are transcribed from teammates' committed artifacts:
#   M1 -> Barshon's/M1/results_M1.json
#   M4 -> executed output of Barshon's/M4/M4_ast_backbone.ipynb
# (M4's results_M4.json lives in Barshon's Google Drive, not in the repo.)
# Re-check them against the source files before pasting into the manuscript.

REFERENCE_ROWS = [
    {"model": "M1 (CNN, provisional)", "acc": 0.5407, "f1": 0.4844, "se": 0.5801,
     "sp": 0.8561, "icbhi": 0.7181, "params": 421_732, "size_mb": 1.61, "ms": None},
    {"model": "M4 (AST, pretrained)", "acc": 0.5528, "f1": 0.4109, "se": 0.4385,
     "sp": 0.8334, "icbhi": 0.6359, "params": 86_385_668, "size_mb": 988.85, "ms": 86.74},
]

m2_row = {
    "model": f"M2 (CNN, tuned d{eval_config['depth']}/w{eval_config['base_width']})",
    "acc": final_metrics["accuracy"], "f1": final_metrics["f1_macro"],
    "se": final_metrics["recall_macro"], "sp": final_metrics["specificity_macro"],
    "icbhi": final_metrics["icbhi_score"], "params": total_params,
    "size_mb": model_size_mb, "ms": inference_ms,
}

print("=" * 104)
print("M12 BACKBONE-ARCHITECTURE ABLATION TABLE (all rows: ICBHI 2017, official 60/40, "
      "class-weighted CE, no augmentation)")
print("=" * 104)
hdr = (f"{'Model':<34}{'Acc':>8}{'F1':>8}{'Se':>8}{'Sp':>8}{'ICBHI':>9}"
       f"{'Params':>14}{'Size MB':>10}{'ms/sample':>11}")
print(hdr)
print("-" * 104)
for r in REFERENCE_ROWS + [m2_row]:
    ms_str = f"{r['ms']:.2f}" if r["ms"] is not None else "n/a"
    print(f"{r['model']:<34}{r['acc']:>8.4f}{r['f1']:>8.4f}{r['se']:>8.4f}{r['sp']:>8.4f}"
          f"{r['icbhi']:>9.4f}{r['params']:>14,}{r['size_mb']:>10.2f}{ms_str:>11}")
print("=" * 104)

# ---- Did tuning actually beat the provisional model? ----
m1_icbhi = REFERENCE_ROWS[0]["icbhi"]
delta = final_metrics["icbhi_score"] - m1_icbhi
print(f"\nM2 vs. M1 (the point of this run): ICBHI {final_metrics['icbhi_score']:.4f} "
      f"vs {m1_icbhi:.4f}  =>  {delta:+.4f}")
if sweep_results:
    spread = sweep_results[0]["std_icbhi"]
    print(f"Sweep cross-validation std for the winning config: +/- {spread:.4f}")
    if abs(delta) < spread:
        print("NOTE: the gap is smaller than the CV fold-to-fold spread. Report it as "
              "'comparable', not as an improvement — this is exactly the kind of claim "
              "a reviewer will check.")

print(f"\nWinning configuration : {eval_config}")
print(f"Best epoch            : {best_epoch} of {CFG['num_epochs']}")
print(f"Total training time   : {final_run['total_time_s'] / 60:.1f} min on {GPU_NAME}")
print(f"\nArtifacts written to  : {CFG['results_dir']}")
print(f"Best checkpoint       : {best_path}")

---

### What to do with these outputs

1. **Download the ZIP bundle** from the handoff cell (§11.D) before closing the Colab session —
   `/content` is wiped, and only the Drive copy survives.
2. **Commit** `results_M2.json` and the four PNGs into `Asif's/M2/` in the repo. The `.pth` file is
   gitignored (`.gitignore:27`), so upload it to Drive and record the link in a
   `Best_model_pth_file Link.txt` next to the notebook — the same convention Barshon used for M4.
3. **Send `results_M2.json` to Barshon.** M12 currently justifies the backbone choice on two
   candidates (M1 and M4), but `Model_Training_Reference.md:114` requires M2, M3, and M4. This file
   is one of the two missing rows.

### Known caveats to carry into the write-up

- **M1's ICBHI score is not a like-for-like comparison target.** M1 selected its best checkpoint on
  the *test* split (as does this notebook, following the same protocol), so both numbers are
  optimistically biased in the same direction. The comparison between them is fair; the absolute
  values should not be presented as clean held-out estimates. The k-fold spread in
  `M2_hp_sweep.json` is the more honest generalisation estimate.
- **M3 is still missing.** Once it exists, the `backbone_architecture` group has all three rows and
  M12's decision can be re-derived from data instead of re-asserted.